# CycloSafe - Cyclone Detection Retraining & Fine-Tuning (V4)

**Goal:** Fine-tune `best_cyclone_detector_finalModel.keras` on 160,000+ images with **9,000 Hard Negatives** (Tea cups/latte art, Ocean sinkholes, Black holes, Von Kármán cloud vortexes, Disorganized thunderstorm clouds, and Calderas) so the model natively rejects false positives without any manual code conditions!

## Step 1: Mount Google Drive & Import Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, cv2, zipfile, shutil, time, random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, callbacks, optimizers
import matplotlib.pyplot as plt

print(f'TensorFlow Version: {tf.__version__}')
print(f'GPU Available: {tf.config.list_physical_devices("GPU")}')

# Config
DRIVE_SAVE_DIR = '/content/drive/MyDrive/CycloSafe_Models'
BEST_MODEL_PATH = os.path.join(DRIVE_SAVE_DIR, 'best_cyclone_detector_finalModel.keras')
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 64
SEED = 42


## Step 2: Extract 150k Dataset Partitions

In [ ]:
EXTRACT_DIR = '/content/dataset_v4'
DRIVE_ZIPS = [
    '/content/drive/MyDrive/dataset_part1_65k_cyclone.zip',
    '/content/drive/MyDrive/dataset_part2_85k_nocyclone.zip'
]

# Extract if not present
if not os.path.exists(os.path.join(EXTRACT_DIR, 'Cyclone')) or not os.path.exists(os.path.join(EXTRACT_DIR, 'No_Cyclone')):
    print('Extracting dataset partitions from Google Drive...')
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    for drive_zip in DRIVE_ZIPS:
        if os.path.exists(drive_zip):
            local_zip = f'/content/{os.path.basename(drive_zip)}'
            print(f'Copying {drive_zip} to local Colab NVMe storage...')
            shutil.copy(drive_zip, local_zip)
            print(f'Extracting {local_zip}...')
            with zipfile.ZipFile(local_zip, 'r') as zf:
                zf.extractall(EXTRACT_DIR)
            os.remove(local_zip)
            print(f'Finished extracting {drive_zip}!')
else:
    print('Dataset already extracted in Colab!')


## Step 3: Inject 9,000 Hard Negatives into No_Cyclone

In [ ]:
NO_CYCLONE_DIR = os.path.join(EXTRACT_DIR, 'No_Cyclone')
CYCLONE_DIR = os.path.join(EXTRACT_DIR, 'Cyclone')
os.makedirs(NO_CYCLONE_DIR, exist_ok=True)
os.makedirs(CYCLONE_DIR, exist_ok=True)
# Extract old 10k Synthetic_Hard_Negatives from Drive (prevents catastrophic forgetting)
SYNTH_OLD_ZIP = '/content/drive/MyDrive/Synthetic_Hard_Negatives.zip'
if os.path.exists(SYNTH_OLD_ZIP):
    print('Found Synthetic_Hard_Negatives.zip in Google Drive! Extracting into No_Cyclone...')
    with zipfile.ZipFile(SYNTH_OLD_ZIP, 'r') as zf:
        for member in zf.infolist():
            if not member.is_dir() and member.filename.endswith(('.jpg', '.png', '.jpeg')):
                member.filename = os.path.basename(member.filename)
                zf.extract(member, NO_CYCLONE_DIR)
    print('10k Old Synthetic Hard Negatives successfully added to No_Cyclone!')

# Check if hard_negatives_9k.zip is in Google Drive, otherwise generate directly in Colab (takes ~15s)
DRIVE_HARD_ZIP = '/content/drive/MyDrive/hard_negatives_9k.zip'
if os.path.exists(DRIVE_HARD_ZIP):
    print('Found hard_negatives_9k.zip in Google Drive! Extracting...')
    with zipfile.ZipFile(DRIVE_HARD_ZIP, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
else:
    print('Generating 9,000 Hard Negatives directly in Colab...')
    t0 = time.time()
    
    # 1. Tea Cups & Latte Art (1,500)
    print('  [1/6] 1,500 Tea Cups & Latte Art Swirls...')
    for i in range(1500):
        img = np.zeros((224, 224, 3), dtype=np.uint8)
        bg_val = random.randint(15, 45)
        img[:, :] = [bg_val, bg_val + random.randint(0, 8), bg_val + random.randint(0, 15)]
        cx, cy, cup_r = random.randint(105, 119), random.randint(105, 119), random.randint(85, 100)
        cv2.circle(img, (cx, cy), cup_r + 8, (random.randint(220, 255), random.randint(220, 255), random.randint(220, 255)), -1)
        liquid_b, liquid_g, liquid_r = random.randint(10, 35), random.randint(60, 120), random.randint(140, 210)
        cv2.circle(img, (cx, cy), cup_r, (liquid_b, liquid_g, liquid_r), -1)
        for r in range(12, cup_r - 10, int(cup_r / random.randint(4, 7))):
            cv2.ellipse(img, (cx, cy), (r, int(r * 0.9)), random.randint(0, 180), 0, 360, (235, 235, 235), random.randint(2, 4))
        cv2.circle(img, (cx, cy), random.randint(8, 20), (int(liquid_b * 0.4), int(liquid_g * 0.4), int(liquid_r * 0.4)), -1)
        cv2.imwrite(os.path.join(NO_CYCLONE_DIR, f'colab_teacup_{i:04d}.jpg'), cv2.GaussianBlur(img, (5, 5), 0))
        
    # 2. Ocean Sinkholes (1,500)
    print('  [2/6] 1,500 Ocean Blue Holes & Sinkholes...')
    for i in range(1500):
        img = np.zeros((224, 224, 3), dtype=np.uint8)
        base_b, base_g, base_r = random.randint(140, 210), random.randint(70, 125), random.randint(0, 30)
        img[:, :] = [base_b, base_g, base_r]
        cx, cy, max_r, hole_r = random.randint(100, 124), random.randint(100, 124), random.randint(65, 95), random.randint(18, 38)
        for r in range(max_r, 0, -2):
            factor = (r / max_r) ** 0.8
            cv2.circle(img, (cx, cy), r, (int(base_b * factor), int(base_g * factor), int(base_r * factor)), -1)
        cv2.circle(img, (cx, cy), hole_r, (random.randint(5, 25), random.randint(5, 20), 0), -1)
        for _ in range(random.randint(3, 6)):
            cv2.circle(img, (cx, cy), random.randint(hole_r + 6, max_r + 15), (230, 230, 230), random.randint(2, 4))
        cv2.imwrite(os.path.join(NO_CYCLONE_DIR, f'colab_sinkhole_{i:04d}.jpg'), cv2.GaussianBlur(img, (7, 7), 0))
        
    # 3. Black Holes & Cosmic Spirals (1,000)
    print('  [3/6] 1,000 Black Holes & Galaxy Spirals...')
    for i in range(1000):
        img = np.zeros((224, 224, 3), dtype=np.uint8)
        for _ in range(random.randint(100, 200)):
            img[random.randint(0, 223), random.randint(0, 223)] = [220, 220, 220]
        cx, cy, hole_r = random.randint(105, 119), random.randint(105, 119), random.randint(25, 45)
        palette = random.choice([(100, 160, 240), (220, 140, 150), (200, 80, 200)])
        for r in range(hole_r + 5, hole_r + 50, 3):
            alpha = (1.0 - (r - hole_r) / 50.0) ** 0.6
            cv2.ellipse(img, (cx, cy), (r, int(r * 0.7)), random.randint(-40, 40), 0, 360, (int(palette[0] * alpha), int(palette[1] * alpha), int(palette[2] * alpha)), 2)
        cv2.circle(img, (cx, cy), hole_r, (0, 0, 0), -1)
        cv2.imwrite(os.path.join(NO_CYCLONE_DIR, f'colab_blackhole_{i:04d}.jpg'), cv2.GaussianBlur(img, (5, 5), 0))
        
    # 4. Satellite Cloud Eddies / Von Karman (2,000)
    print('  [4/6] 2,000 Satellite Cloud Eddies & Von Kármán Streets...')
    for i in range(2000):
        img = np.zeros((224, 224, 3), dtype=np.uint8)
        img[:, :] = [random.randint(70, 130), random.randint(45, 90), random.randint(15, 40)]
        ix, iy = random.randint(30, 60), random.randint(90, 130)
        cv2.circle(img, (ix, iy), random.randint(8, 16), (40, 90, 50), -1)
        for step in range(1, 6):
            ex, ey = ix + step * random.randint(28, 36), iy + ((-1) ** step) * random.randint(15, 30)
            if 0 <= ex < 224 and 0 <= ey < 224:
                cv2.ellipse(img, (ex, ey), (random.randint(14, 26), random.randint(10, 18)), step * 45, 0, 310, (225, 225, 225), random.randint(3, 5))
        cv2.imwrite(os.path.join(NO_CYCLONE_DIR, f'colab_cloud_eddy_{i:04d}.jpg'), cv2.GaussianBlur(img, (7, 7), 0))
        
    # 5. Disorganized Clouds / ITCZ (2,000)
    print('  [5/6] 2,000 Disorganized Tropical Clouds (ITCZ)...')
    for i in range(2000):
        img = np.zeros((224, 224, 3), dtype=np.uint8)
        img[:, :] = [random.randint(80, 140), random.randint(50, 95), random.randint(20, 45)]
        for _ in range(random.randint(8, 16)):
            cv2.ellipse(img, (random.randint(20, 204), random.randint(20, 204)), (random.randint(25, 65), random.randint(15, 45)), random.randint(0, 180), 0, 360, (230, 230, 230), -1)
        cv2.imwrite(os.path.join(NO_CYCLONE_DIR, f'colab_itcz_{i:04d}.jpg'), cv2.GaussianBlur(img, (9, 9), 0))
        
    # 6. Geological Circles & Calderas (1,000)
    print('  [6/6] 1,000 Geological Circles & Calderas...')
    for i in range(1000):
        img = np.zeros((224, 224, 3), dtype=np.uint8)
        sand_b, sand_g, sand_r = random.randint(40, 85), random.randint(90, 150), random.randint(160, 230)
        img[:, :] = [sand_b, sand_g, sand_r]
        cx, cy = random.randint(100, 124), random.randint(100, 124)
        for r in range(15, 95, 12):
            cv2.circle(img, (cx, cy), r, (max(0, sand_b - 20), max(0, sand_g - 35), max(0, sand_r - 45)), random.randint(3, 6))
        cv2.circle(img, (cx, cy), random.randint(10, 20), (sand_b - 20, sand_g - 20, sand_r - 20), -1)
        cv2.imwrite(os.path.join(NO_CYCLONE_DIR, f'colab_geology_{i:04d}.jpg'), cv2.GaussianBlur(img, (5, 5), 0))
        
    print(f'Done in {time.time() - t0:.2f}s!')

c_count = len(os.listdir(CYCLONE_DIR))
nc_count = len(os.listdir(NO_CYCLONE_DIR))
print('==================================================')
print(f'DATASET READY: No_Cyclone = {nc_count:,} | Cyclone = {c_count:,} | Total = {c_count + nc_count:,}')
print('==================================================')


## Step 4: Load Existing Model & Fine-Tune with Auto-Save

In [ ]:
MODEL_PATH = '/content/drive/MyDrive/best_cyclone_detector_finalModel.keras'
if not os.path.exists(MODEL_PATH):
    MODEL_PATH = '/content/drive/MyDrive/best_cyclone_detector.keras'

print(f'Loading existing model from: {MODEL_PATH}')
model = tf.keras.models.load_model(MODEL_PATH)
model.trainable = True

from tensorflow.keras.applications.efficientnet import preprocess_input
# Training data generator with heavy augmentations to generalize well
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    validation_split=0.15
)

# Validation data generator (NO augmentations, only preprocessing for pure metric evaluation)
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.15
)

train_gen = train_datagen.flow_from_directory(
    EXTRACT_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True,
    seed=SEED
)

val_gen = val_datagen.flow_from_directory(
    EXTRACT_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False,
    seed=SEED
)

# Auto-save after EVERY epoch directly to Google Drive
class SaveEveryEpoch(callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        drive_root_path = '/content/drive/MyDrive/best_cyclone_detector_finalModel.keras'
        self.model.save(drive_root_path)
        models_folder_path = os.path.join(DRIVE_SAVE_DIR, 'best_cyclone_detector_finalModel.keras')
        self.model.save(models_folder_path)
        print(f'\n[AutoSave Epoch {epoch + 1}] Model successfully backed up to Google Drive!')
        print(f'  -> Saved to: {drive_root_path}')

# Small learning rate (1e-5) preserves all existing knowledge while mastering hard negatives
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

callbacks_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1),
    SaveEveryEpoch()
]

EPOCHS = 6
print(f'Starting fine-tuning for {EPOCHS} epochs...')
t0 = time.time()
history = model.fit(train_gen, epochs=EPOCHS, validation_data=val_gen, callbacks=callbacks_list, verbose=1)
print(f'Retraining complete in {(time.time()-t0)/60:.2f} minutes!')


## Step 5: Save & Download Final Model

In [ ]:
FINAL_SAVE_PATH = '/content/drive/MyDrive/best_cyclone_detector_finalModel.keras'
model.save(FINAL_SAVE_PATH)
print(f'Final retrained model saved to Google Drive: {FINAL_SAVE_PATH}')

from google.colab import files
print('Downloading updated model to your local computer...')
files.download(FINAL_SAVE_PATH)
